# Choosing qubits: benchmarking a processor you cannot see inside

*Part of the QUEST Systems, Hardware & Engineering series*

Every other notebook in this series picks a backend by name and submits. That hides a decision. A processor is not uniform: on any real device some qubits are markedly worse than others, the ranking changes between calibrations, and a circuit placed on the wrong four qubits can return noise while the same circuit on the right four returns a usable result. Nothing in the algorithm changes. Only the placement does.

The obvious way to choose is to read the vendor's calibration data. That works within one vendor's stack and stops working the moment you use several. A unified interface has to expose what every backend can supply, so `device.metadata()` gives you qubit count, basis gates, status and queue depth, and no per-qubit error rates. Across vendors the numbers would not be comparable anyway: they are measured by different protocols, reported against different definitions of fidelity, and refreshed on different schedules.

So we measure the device ourselves, with circuits that need no calibration file, no tomography, and no reference device. Mirror circuits give a number that means the same thing on a superconducting chip and on an ion trap, because it is defined by the circuits we ran rather than by the vendor's characterisation. We validate the method against a simulator with a known injected error rate, run it on three processors, rank the qubits of one of them, then take the BB84 circuit from the Cryptography & Security series and show what the ranking is worth.

**Learning objectives.** By the end of this notebook you will:

1. State what a unified device interface can and cannot tell you about qubit quality.
2. Build randomized mirror circuits and predict their ideal output without simulating them.
3. Validate a benchmark by injecting a known error rate and recovering it.
4. Explain why a benchmark must be protected from the compiler, and what happens when it is not.
5. Rank the qubits of a real processor by measured survival and choose a subset from that ranking.
6. Measure what the choice is worth by running a real protocol on the best and worst subsets.

**What to bring in.** Single-qubit and two-qubit gates, the Clifford group at the level of "H and S and CNOT generate it", and how a depolarising channel acts. The BB84 circuit from the Cryptography & Security series is reused at the end but restated here, so that notebook is useful background rather than a prerequisite.

**Credit budget.** Simulator work is free. Hardware is 24 jobs at 500 shots for the device sweep, 6 jobs at 1,000 shots for the qubit ranking, and 2 jobs at 1,000 shots for the payoff, so about 20,000 shots in 32 jobs. At typical rates this is a few dollars in credits. The job count matters more than the shot count for wall-clock time.

## What the platform actually tells you

Start by looking at what is on offer, because the gap between that and what you need is the reason for the rest of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Clifford, Pauli
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110

rng = np.random.default_rng(11)
sim = AerSimulator(seed_simulator=42)

print("Setup complete.")

In [ ]:
provider = QbraidProvider()

# Instructor: update these IDs to match currently available hardware in your account.
BACKENDS = {
    'Rigetti Ankaa-3': 'rigetti_ankaa_3',
    'IQM Garnet':      'iqm_garnet',
    'AQT Marmot':      'aqt_marmot',
}

COLORS = {
    'Rigetti Ankaa-3': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT Marmot':      '#2d7a4f',
}

devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}

meta = pd.DataFrame([
    {'Backend': name, **{k: v for k, v in dev.metadata().items()
                         if k in ('num_qubits', 'provider_name', 'paradigm',
                                  'status', 'queue_depth', 'average_queue_time')}}
    for name, dev in devices.items()
]).set_index('Backend')
meta

Qubit count, vendor, modality, availability, queue depth. Everything you need to decide whether a job will run, and nothing about whether it will run *well*. There is no per-qubit error rate here, no coupling map, no readout fidelity, and no timestamp saying when any of it was last measured.

This is not an oversight in the interface. It is what survives translation between vendors. A superconducting chip reports two-qubit gate error from randomized benchmarking on specific pairs; an ion trap with all-to-all coupling has no pairs to report and a different dominant error mechanism. Averaging them into one field would produce a number that compares nothing.

## Mirror circuits

A benchmark needs a circuit whose correct output you know without simulating it, since simulating it is exactly what a useful-sized quantum circuit forbids. Mirror circuits get this for free. Run a circuit $C$, then run $C^{\dagger}$. The composition is the identity, so whatever you put in comes back out, and any deviation is error. The ideal output is known at any width and any depth, at no classical cost.

Run naively this overestimates fidelity. Coherent errors, the systematic kind from miscalibrated pulses, tend to undo themselves when the circuit is inverted: an over-rotation in $C$ becomes an equal and opposite over-rotation in $C^{\dagger}$. The device looks better than it is.

The fix is a random Pauli inserted between the two halves. Since $C$ is Clifford, $C^{\dagger} P C$ is another Pauli $P'$, so the ideal output is still a single computational basis state, just not $\lvert 0 \dots 0 \rangle$ any more. We compute which one classically, which is cheap because propagating a Pauli through a Clifford is cheap. The insert breaks the symmetry that let coherent errors cancel, and randomising over $P$ converts them into an effective stochastic error that the decay curve can see. This is the construction from Proctor and co-workers' randomized mirror circuits, reduced to its essentials.

In [ ]:
CLIFF1Q = ['id', 'x', 'y', 'z', 'h', 's', 'sdg']


def mirror_circuit(nq, depth, rng, pairs=None, qubits=None):
    """
    A randomized mirror circuit on `nq` qubits.

    Returns (circuit, ideal_bitstring). The circuit is C, then a random Pauli,
    then C-dagger. Because C is Clifford, the net operation is a Pauli, so the
    ideal output is one basis state, computed here rather than simulated.
    """
    qubits = list(range(nq)) if qubits is None else list(qubits)

    front = QuantumCircuit(nq)
    for _ in range(depth):
        for q in qubits:
            getattr(front, rng.choice(CLIFF1Q))(q)
        front.barrier()                       # keep the layers from merging
        if pairs:
            for a, b in pairs:
                front.cx(a, b)
            front.barrier()

    z_part = rng.integers(0, 2, nq).astype(bool)
    x_part = rng.integers(0, 2, nq).astype(bool)
    pauli = Pauli((z_part, x_part))

    qc = QuantumCircuit(nq, nq)
    qc.compose(front, inplace=True)
    qc.barrier()
    qc.compose(pauli.to_instruction(), range(nq), inplace=True)
    qc.barrier()
    qc.compose(front.inverse(), inplace=True)
    qc.measure(range(nq), range(nq))

    # Heisenberg frame: P -> C-dagger P C, which is what acts on |0...0>.
    net = pauli.evolve(Clifford(front), frame='h')
    ideal = ''.join('1' if b else '0' for b in reversed(net.x))
    return qc, ideal


qc_demo, ideal_demo = mirror_circuit(3, 2, np.random.default_rng(0), pairs=[(0, 1)])
print(f"predicted ideal output: {ideal_demo}")
qc_demo.draw('mpl', fold=120)

In [ ]:
# The prediction is a claim about the circuit. Check it before relying on it.
ok = 0
trials = 40
for _ in range(trials):
    nq = int(rng.integers(1, 6))
    pairs = [(i, i + 1) for i in range(nq - 1)] if nq > 1 else None
    qc, ideal = mirror_circuit(nq, int(rng.integers(1, 8)), rng, pairs)
    counts = sim.run(transpile(qc, sim, optimization_level=0), shots=200).result().get_counts()
    ok += (len(counts) == 1 and list(counts)[0] == ideal)

print(f"predicted output matched noiseless simulation in {ok}/{trials} random instances")

## The compiler will happily destroy your benchmark

A mirror circuit is a circuit followed by its own inverse. That is precisely the pattern an optimising compiler exists to collapse. Left alone, the transpiler recognises the structure, resynthesises it, and hands the device a few gates instead of a few hundred. The device then reports an excellent survival probability, because it was asked to run almost nothing.

The barriers inside `mirror_circuit` are what prevent this. It is worth seeing the size of the effect.

In [ ]:
def cx_after_transpile(qc, level, barriers=True):
    if not barriers:
        qc = qc.copy()
        qc.data = [d for d in qc.data if d.operation.name != 'barrier']
    return transpile(qc, basis_gates=['cx', 'rz', 'sx', 'x'],
                     optimization_level=level, seed_transpiler=1).count_ops().get('cx', 0)


rows = []
for depth in (2, 4, 8, 16):
    qc, _ = mirror_circuit(4, depth, np.random.default_rng(depth), pairs=[(0, 1), (2, 3)])
    rows.append({
        'depth': depth,
        'CX as written': qc.count_ops().get('cx', 0),
        'CX, barriers, opt 3': cx_after_transpile(qc, 3, barriers=True),
        'CX, no barriers, opt 3': cx_after_transpile(qc, 3, barriers=False),
    })

pd.DataFrame(rows).set_index('depth')

With barriers the circuit reaches the device intact at every depth. Without them the compiler collapses each mirrored two-qubit block back to an optimal synthesis of the identity and the depth sweep flattens into a constant. A benchmark that does this measures the compiler, not the processor, and it fails silently: the numbers look good.

The same argument applies in reverse when you are running real work, where you want every optimisation the compiler has. The compilation notebook in this series takes that side of the trade.

## Does the benchmark measure what it claims?

Before trusting the number on hardware, we check it against a device whose error rate we set ourselves. Inject a known two-qubit depolarising rate into a simulator, run the sweep, fit the decay, and see whether the fit returns the injected value.

The survival probability of a mirror circuit falls exponentially in the number of noisy gates, on top of a floor at $2^{-n}$ where the state is fully randomised:

$$S(d) = A\, f^{\,g(d)} + 2^{-n}$$

with $g(d)$ the gate count at depth $d$. For a two-qubit depolarising channel of strength $p$ the decay constant is the average gate infidelity $3p/4$, not $p$ itself, because a depolarising channel leaves the state alone a quarter of the time by accident. Getting that factor right is the difference between a benchmark and a plausible-looking number.

In [ ]:
def survival(circuit_fn, backend, depths, instances, shots, nq):
    """Average survival probability over random mirror instances at each depth."""
    out = []
    for d in depths:
        tot = 0.0
        for _ in range(instances):
            qc, ideal = circuit_fn(d)
            counts = backend.run(transpile(qc, backend, optimization_level=0),
                                 shots=shots).result().get_counts()
            tot += counts.get(ideal, 0) / shots
        out.append(tot / instances)
    return np.array(out)


def fit_infidelity(depths, surv, nq, gates_per_depth):
    """Fit S(d) = A f^g(d) + 2^-n and return the per-gate infidelity 1 - f."""
    y = surv - 2.0 ** -nq
    keep = y > 1e-3
    if keep.sum() < 2:
        return np.nan
    slope = np.polyfit(np.asarray(depths)[keep] * gates_per_depth,
                       np.log(y[keep]), 1)[0]
    return 1 - np.exp(slope)


PAIRS = [(0, 1), (2, 3)]
DEPTHS = [1, 2, 4, 8, 16]
NQ = 4
GATES_PER_DEPTH = 2 * len(PAIRS)          # each layer appears twice, once mirrored

rows = []
for p_inj in (0.005, 0.02, 0.05):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p_inj, 2), ['cx'])
    noisy = AerSimulator(noise_model=nm, seed_simulator=2)

    surv = survival(lambda d: mirror_circuit(NQ, d, rng, PAIRS),
                    noisy, DEPTHS, instances=20, shots=400, nq=NQ)
    est = fit_infidelity(DEPTHS, surv, NQ, GATES_PER_DEPTH)
    rows.append({'injected p': p_inj, 'expected 3p/4': round(3 * p_inj / 4, 4),
                 'recovered': round(est, 4), 'ratio': round(est / (3 * p_inj / 4), 2),
                 'survival': [round(s, 3) for s in surv]})

pd.DataFrame(rows).set_index('injected p')

Recovery to within a few percent across an order of magnitude in error rate. The residual bias is real and comes from the crude two-parameter fit and from attributing all of the decay to the two-qubit gates when the single-qubit layers contribute too. For ranking qubits and devices that bias is irrelevant, because it applies equally to everything being ranked. For quoting an absolute fidelity it is not, and the proper treatment fits the single-qubit and two-qubit contributions separately.

## The same measurement on three processors

Now run the sweep on hardware. The number that comes back is an error per two-qubit layer *as the platform delivers it to you*, which includes whatever routing the compiler had to insert to satisfy the device's connectivity. That is deliberate. We have no coupling map, and the quantity a user of a unified platform actually experiences is the compiled circuit, not the abstract one. A device with excellent gates and awkward connectivity should and does score worse here.

**Note on queue times.** This is 24 hardware jobs. Depending on device load the submission cell may take minutes to hours.

In [ ]:
HW_DEPTHS = [1, 2, 4, 8]
HW_INSTANCES = 2
HW_SHOTS = 500

hw_survival = {}
for name, device in devices.items():
    surv = []
    for d in HW_DEPTHS:
        tot = 0.0
        for _ in range(HW_INSTANCES):
            qc, ideal = mirror_circuit(NQ, d, rng, PAIRS)
            counts = device.run(qc, shots=HW_SHOTS).result().data.get_counts()
            tot += counts.get(ideal, 0) / HW_SHOTS
        surv.append(tot / HW_INSTANCES)
    hw_survival[name] = np.array(surv)
    print(f"{name:18s} survival {[round(s, 3) for s in surv]}")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))

xs = np.array(HW_DEPTHS) * GATES_PER_DEPTH
ax.axhline(2.0 ** -NQ, color='gray', linestyle=':', linewidth=2,
           label=f'Fully randomised ($2^{{-{NQ}}}$)')
ax.axhline(1.0, color='k', linestyle='--', linewidth=2, alpha=0.6, label='Ideal')

for name in BACKENDS:
    ax.plot(xs, hw_survival[name], 'o-', color=COLORS[name], label=name,
            markersize=11, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('two-qubit gates in the circuit')
ax.set_ylabel('survival probability')
ax.set_title('Randomized mirror circuits on three processors')
ax.set_ylim(0, 1.08)
ax.grid(alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

summary = pd.DataFrame([
    {'Backend': name,
     'Survival at depth 1': f'{hw_survival[name][0]:.3f}',
     'Survival at depth 8': f'{hw_survival[name][-1]:.3f}',
     'Infidelity per 2Q gate':
         f'{fit_infidelity(HW_DEPTHS, hw_survival[name], NQ, GATES_PER_DEPTH):.4f}'}
    for name in BACKENDS
]).set_index('Backend')
summary

One number per device, measured the same way on each, on circuits we chose rather than circuits the vendor chose. It is not a replacement for the vendor's own characterisation, which is more careful and more granular. It is comparable across vendors, which the vendor numbers are not.

## Ranking the qubits of one device

The device-level number hides the spread within a chip. Single-qubit mirror circuits expose it, and they cost almost nothing: run one circuit across the whole register, with an independent random mirror on every qubit at once, and read a separate survival probability per qubit out of the same shot record. No two-qubit gates means no routing and no connectivity assumptions.

In [ ]:
RANK_TARGET = 'IQM Garnet'          # instructor: pick a device you have access to
RANK_DEPTHS = [4, 16, 64]
RANK_INSTANCES = 2
RANK_SHOTS = 1000


def per_qubit_survival(counts, ideal, shots, width):
    """Survival for each qubit separately, from one shot record."""
    hits = np.zeros(width)
    for bits, num in counts.items():
        b = bits.replace(' ', '')
        for q in range(width):
            if b[width - 1 - q] == ideal[width - 1 - q]:
                hits[q] += num
    return hits / shots

In [ ]:
width = devices[RANK_TARGET].num_qubits
print(f"{RANK_TARGET}: ranking {width} qubits")

qubit_scores = np.zeros(width)
n_meas = 0

for d in RANK_DEPTHS:
    for _ in range(RANK_INSTANCES):
        qc, ideal = mirror_circuit(width, d, rng, pairs=None)
        counts = devices[RANK_TARGET].run(qc, shots=RANK_SHOTS).result().data.get_counts()
        qubit_scores += per_qubit_survival(counts, ideal, RANK_SHOTS, width)
        n_meas += 1

qubit_scores /= n_meas
order = np.argsort(-qubit_scores)
print(f"best 4 qubits:  {order[:4].tolist()}  survival {qubit_scores[order[:4]].round(3).tolist()}")
print(f"worst 4 qubits: {order[-4:].tolist()}  survival {qubit_scores[order[-4:]].round(3).tolist()}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

colors = ['#2d7a4f' if q in order[:4] else '#a02580' if q in order[-4:] else '#1a5285'
          for q in range(width)]
ax.bar(range(width), qubit_scores, color=colors, width=0.7)
ax.axhline(qubit_scores.mean(), color='k', linestyle='--', linewidth=2, alpha=0.6,
           label=f'device mean ({qubit_scores.mean():.3f})')
ax.set_xlabel('physical qubit index')
ax.set_ylabel('mean survival')
ax.set_title(f'Per-qubit survival on {RANK_TARGET}: green best four, magenta worst four')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3, axis='y')
ax.legend()
plt.tight_layout()
plt.show()

## What the ranking is worth

A ranking is only interesting if acting on it changes a result. We take the BB84 circuit from the Cryptography & Security series, which prepares the four protocol states on four qubits and measures each in its own basis, and run it twice on the same device in the same session: once on the four best qubits, once on the four worst. The circuit uses no two-qubit gates, so placement is the only variable and routing cannot confound the comparison.

The quantity it returns is the protocol's error rate, and BB84 aborts above 11%.

In [ ]:
Z_BASIS, X_BASIS = 0, 1
BB84_STATES = [(0, Z_BASIS), (1, Z_BASIS), (0, X_BASIS), (1, X_BASIS)]


def bb84_on_qubits(physical, width):
    """The four BB84 states, one per chosen physical qubit, each measured in its own basis."""
    qc = QuantumCircuit(width, len(BB84_STATES))
    for slot, (bit, basis) in enumerate(BB84_STATES):
        q = int(physical[slot])
        if bit:
            qc.x(q)
        if basis == X_BASIS:
            qc.h(q)
    qc.barrier()
    for slot, (bit, basis) in enumerate(BB84_STATES):
        q = int(physical[slot])
        if basis == X_BASIS:
            qc.h(q)
        qc.measure(q, slot)
    return qc


def qber_from(counts, shots):
    errs = 0
    for bits, num in counts.items():
        b = bits.replace(' ', '')
        for slot, (bit, _) in enumerate(BB84_STATES):
            if int(b[len(BB84_STATES) - 1 - slot]) != bit:
                errs += num
    return errs / (shots * len(BB84_STATES))


# Sanity check on the simulator: the ideal QBER must be zero for any placement.
for placement in (order[:4], order[-4:]):
    qc = bb84_on_qubits(placement, width)
    counts = sim.run(transpile(qc, sim, optimization_level=0), shots=2000).result().get_counts()
    print(f"ideal QBER on qubits {placement.tolist()}: {qber_from(counts, 2000):.4f}")

In [ ]:
PAYOFF_SHOTS = 1000
placements = {'best four': order[:4], 'worst four': order[-4:]}

payoff = {}
for label, placement in placements.items():
    qc = bb84_on_qubits(placement, width)
    counts = devices[RANK_TARGET].run(qc, shots=PAYOFF_SHOTS).result().data.get_counts()
    payoff[label] = qber_from(counts, PAYOFF_SHOTS)
    print(f"{label:11s} qubits {placement.tolist()}  QBER = {payoff[label]:.4f}")

In [ ]:
def key_rate(q):
    """Shor-Preskill asymptotic rate r = 1 - 2h(Q), floored at zero."""
    if q <= 0:
        return 1.0
    if q >= 0.5:
        return 0.0
    h = -q * np.log2(q) - (1 - q) * np.log2(1 - q)
    return max(0.0, 1 - 2 * h)


rows = []
for label, placement in placements.items():
    q = payoff[label]
    rows.append({
        'Placement': label,
        'Qubits': str(placement.tolist()),
        'Mean mirror survival': f'{qubit_scores[placement].mean():.3f}',
        'Measured QBER': f'{q:.4f}',
        'Key rate': f'{key_rate(q):.3f}',
        'Secure bits per 1000 sifted': f'{1000 * key_rate(q):.0f}',
        'BB84 verdict': 'ABORT' if q >= 0.11 else 'ok',
    })

pd.DataFrame(rows).set_index('Placement')

Same device, same circuit, same session, same shot count. The only difference is which four physical qubits ran it, and that choice came from a benchmark that took six jobs to produce.

The row to look at is the last one. If the worst four qubits push the QBER past 11% while the best four stay under it, then placement alone decides whether the protocol produces any key at all, and a paper reporting only "BB84 on device X" has left out the variable that determined its result.

## What this measurement does not tell you

Mirror circuits are cheap, portable and self-verifying, which makes them easy to over-read. Four limits are worth stating.

**They measure a random circuit, not yours.** The survival probability is an average over random Clifford layers. Your circuit is one specific structure, and if its errors are structured, the average will not predict it. The honest use is comparison between placements and between devices, which is what we used it for. Predicting an absolute fidelity for a particular algorithm needs a benchmark shaped like that algorithm.

**They fold everything into one number.** Gate error, readout error, idle decoherence and crosstalk all reduce survival, and the fit cannot separate them. A qubit that ranks poorly here might have excellent gates and terrible readout. That distinction matters if you plan to fix it, and not at all if you plan to avoid the qubit.

**Parallel single-qubit mirrors measure qubits in a crowd.** Running every qubit at once is what makes the ranking affordable, and it means each qubit is measured while its neighbours are active. Crosstalk is therefore included in the score. That is usually what you want, since your real circuit will also run with neighbours active, but it is not the same as an isolated per-qubit fidelity and should not be quoted as one.

**The ranking expires.** Devices are recalibrated on a schedule and drift between calibrations. A ranking measured hours before the run may no longer hold, which is why the payoff above deliberately runs the benchmark and the protocol in the same session. Treat a stored ranking with the same suspicion as a stored calibration file.

The broader point is the one this series keeps arriving at. The device is not a fixed object that the algorithm runs on top of. It is part of the experiment, it varies across its own surface and across time, and choosing which part of it to use is a step in the method rather than an implementation detail. The QAOA and VQE notebooks both leave real performance on the table by taking whatever placement the compiler chose, and the amount left is measurable with the six jobs above.

## Where to go next

- **Separate the error sources.** Run the depth sweep with the two-qubit layers removed, so only single-qubit gates and idling contribute, and subtract. Compare the two-qubit infidelity you get this way against the single-fit number above.
- **Measure readout separately.** Prepare each basis state and measure it immediately, with no mirror at all. Build the confusion matrix per qubit and check how much of the per-qubit ranking is readout rather than gates.
- **Find the crosstalk.** Run single-qubit mirrors on one qubit alone, then on all qubits at once, and take the difference. The gap is what your neighbours cost you, and on some devices it dominates.
- **Rank pairs, not qubits.** Extend the ranking to two-qubit mirrors over candidate pairs, then place a Bell state on the best and worst pair. Report the transpiled two-qubit count alongside, since without a coupling map the compiler may be routing and the comparison is not clean.
- **Track the drift.** Run the ranking once an hour for a day. Plot how the top four changes. Decide from your own data how stale a ranking may be before you rerun it.
- **Feed it back.** Take the QAOA or VQE notebook, benchmark first, pin `initial_layout` to the best subset, and measure whether the final energy improves.

---

**Feedback for the QUEST pedagogy study**

Your feedback informs the IRB-approved QUEST research project on quantum computing pedagogy. Please spend 2 minutes on the following:

1. What was the most useful part of this notebook for your learning?
2. What was the most confusing or under-explained?
3. Which hardware or systems topic would you most want to see analyzed on real hardware next?

Please submit your responses via the QUEST portal or reply to your instructor.